In [88]:
import cv2
import torch
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from collections import defaultdict, Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "MAIN_MODULE" / "src"))
sys.path.insert(0, str(Path.cwd() / "DEPARTMENT_CLASSIFICATION" / "train_model"))

from crop_extraction import CropCandidate, CropScorer
from predict_single import DepartmentPredictor
sys.path.insert(0, str(Path.cwd() / "VLM_MODULE"))
from detect import load_vlm_model, vlm_predict_crops



In [89]:
config = OmegaConf.load('params.yaml')
root = Path.cwd()
video_folder = root / config.main_extraction.input_folder
yolo_path = root / config.main_extraction.model_path
dept_model_path = root / config.department_classifier.model_path
class_names_path = root / "DEPARTMENT_CLASSIFICATION/train_model/models/class_names.json"

In [90]:
video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".webm"}
videos = sorted(p for p in video_folder.iterdir()
                if p.suffix.lower() in video_extensions and not p.name.startswith("~"))
video_path = videos[0]
print(f"Видео: {video_path.name}")

Видео: 25_12-20.mp4


In [91]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(str(yolo_path)).to(device)
crop_scorer = CropScorer(
    config.main_extraction.min_crop_width,
    config.main_extraction.min_crop_height,
    config.main_extraction.sharpness_threshold,
)

def _predict_np(self, img, top_k=3):
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = self.transform(image=img)
    x = t['image'].unsqueeze(0).to(self.device)
    with torch.no_grad():
        p = torch.softmax(self.model(x), dim=1)[0]
    top = torch.topk(p, top_k)
    return [(self.class_names[i.item()], v.item() * 100) for v, i in zip(top.values, top.indices)]

DepartmentPredictor.predict_np = _predict_np
classifier = DepartmentPredictor(str(dept_model_path), str(class_names_path))

Создана efficientnet-b0 со случайными весами
Модель загружена: efficientnet-b0
Классов: 15
Устройство: cuda


In [92]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"{total} frames, {fps:.2f} FPS")

865 frames, 19.96 FPS


In [93]:
seg_size = total // 5
half_window = 15
step = 5
segment_frames = []
for i in range(5):
    mid = i * seg_size + seg_size // 2
    start = max(0, mid - half_window)
    end = min(total - 1, mid + half_window)
    segment_frames.append(list(range(start, end + 1, step)))
for i, frames in enumerate(segment_frames):
    print(f"  Сегмент {i+1}: {len(frames)} кадров ({frames[0]/fps:.1f}с - {frames[-1]/fps:.1f}с)")

  Сегмент 1: 7 кадров (3.6с - 5.1с)
  Сегмент 2: 7 кадров (12.2с - 13.7с)
  Сегмент 3: 7 кадров (20.9с - 22.4с)
  Сегмент 4: 7 кадров (29.6с - 31.1с)
  Сегмент 5: 7 кадров (38.2с - 39.7с)


In [94]:
best: dict[int, list[CropCandidate]] = defaultdict(list)
frame_to_seg = {}
for sid, frames in enumerate(segment_frames):
    for f in frames:
        frame_to_seg[f] = sid
seg_preds = [[] for _ in range(5)]

cap = cv2.VideoCapture(str(video_path))
fi = -1
while True:
    ok, fr = cap.read()
    if not ok:
        break
    fi += 1

    if config.main_extraction.rotate_frames:
        fr = cv2.rotate(fr, cv2.ROTATE_90_COUNTERCLOCKWISE)

    res = yolo.track(source=fr, persist=True, tracker=config.main_extraction.tracker_config,
                     conf=config.main_extraction.conf_threshold, iou=config.main_extraction.iou_threshold, verbose=False)[0]

    boxes = None
    if res.boxes is not None and res.boxes.id is not None:
        boxes = res.boxes.xyxy.cpu().numpy()
        for box, conf, tid in zip(boxes, res.boxes.conf.cpu().numpy(), res.boxes.id.cpu().numpy().astype(int)):
            x1, y1, x2, y2 = map(int, box)
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(fr.shape[1], x2), min(fr.shape[0], y2)
            crop = fr[y1:y2, x1:x2]
            score = crop_scorer.compute_score(crop, conf)
            if score is None:
                continue
            c = CropCandidate(score, crop.copy(), fi, float(conf), [x1, y1, x2, y2])
            best[tid].append(c)
            best[tid].sort(key=lambda x: x.score, reverse=True)
            best[tid] = best[tid][:config.main_extraction.top_k]

    if fi in frame_to_seg:
        sid = frame_to_seg[fi]
        dept, prob = classifier.predict_np(fr)[0]
        seg_preds[sid].append({'frame': fi, 'time': fi / fps, 'department': dept, 'prob': prob})

    if fi % 500 == 0:
        print(f"Frame {fi}/{total}, tracks: {len(best)}")

cap.release()
print(f"Done. Tracks: {len(best)}, crops: {sum(len(v) for v in best.values())}")

Frame 0/865, tracks: 8
Frame 500/865, tracks: 42
Done. Tracks: 63, crops: 63


In [103]:
# Сбор результатов по предсказанию отдела
rows_seg = []
for sid, preds in enumerate(seg_preds):
    for p in preds:
        rows_seg.append({
            'segment': sid + 1,
            'frame': p['frame'],
            'time_sec': round(p['time'], 1),
            'department': p['department'],
            'probability': round(p['prob'], 1),
        })

department = pd.DataFrame(rows_seg)

In [95]:
rows_crops = []
for track_id, candidates in best.items():
    for rank, c in enumerate(candidates):
        rows_crops.append({
            'filename': video_path.name,
            'SYS_track_id': track_id,
            'SYS_rank': rank + 1,
            'SYS_score': round(c.score, 1),
            'SYS_confidence': round(c.confidence, 3),
            'product_name': None, 'price_default': None, 'price_card': None,
            'price_discount': None, 'barcode': None, 'discount_amount': None,
            'id_sku': None, 'print_datetime': None, 'code': None,
            'additional_info': None, 'color': None, 'special_symbols': None,
            'frame_timestamp': int(c.frame_index / fps * 1000),
            'x_min': c.bbox[0], 'y_min': c.bbox[1],
            'x_max': c.bbox[2], 'y_max': c.bbox[3],
            'qr_code_barcode': None, 'price1_qr': None, 'price2_qr': None,
            'price3_qr': None, 'price4_qr': None,
            'wholesale_level_1_count': None, 'wholesale_level_1_price': None,
            'wholesale_level_2_count': None, 'wholesale_level_2_price': None,
            'action_price_qr': None, 'action_code_qr': None,
        })

df_crops = pd.DataFrame(rows_crops)

In [96]:
vlm_model, vlm_processor = load_vlm_model(str(root / 'VLM_MODULE' / 'AVITO'))


Loading weights:   0%|          | 2/729 [00:00<00:43, 16.71it/s]c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 729/729 [00:04<00:00, 166.17it/s]


In [ ]:
# crop_array from best into df_crops
crop_map = {}
for tid, candidates in best.items():
    for rank, c in enumerate(candidates):
        crop_map[(tid, rank + 1)] = c.crop

df_crops['crop_array'] = df_crops.apply(
    lambda r: crop_map.get((r['SYS_track_id'], r['SYS_rank']), None), axis=1)

df_vlm = vlm_predict_crops(df_crops, vlm_model, vlm_processor)

c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  OCR [5/63]
  OCR [10/63]
  OCR [15/63]
  OCR [20/63]
  OCR [25/63]
  OCR [30/63]
  OCR [35/63]
  OCR [40/63]
  OCR [45/63]
  OCR [50/63]
  OCR [55/63]
  OCR [60/63]


,filename,SYS_track_id,SYS_rank,SYS_score,SYS_confidence,product_name,price_default,price_card,price_discount,barcode,...,wholesale_level_2_price,action_price_qr,action_code_qr,price_without_card,price_with_card,promo_price,discount_size,article,layout_code,print_date
0,25_12-20.mp4,1,1,2338.4,0.979,<0x0A>GRILL▁WINE▁,None,None,None,▁12345_678▁,...,None,None,None,▁295▁,▁295▁,▁-▁,▁26%▁,▁12345_678▁,▁12345_678,
1,25_12-20.mp4,2,1,3899.5,0.976,<0x0A>Вино▁ТОРО▁Бланко▁ординар▁|▁Аргентина▁,None,None,None,▁1L▁,...,None,None,None,▁659₽▁,▁659₽▁,▁-21%▁,▁242▁,▁12345_678▁,34:56,
2,25_12-20.mp4,3,1,2799.6,0.970,34:56▁26%▁12345_678▁2023-04-15▁12:34:56,None,None,None,,...,None,None,None,,,,26%,12345_678,,
3,25_12-20.mp4,4,1,2982.8,0.321,<0x0A>Продукты_название,None,None,None,Штрихкод,...,None,None,None,Цена_без_карты,Цена_с_картой,Промо_цена,Размер_скида,Номер_артикул,Код_лайаута,Дата_печати
4,25_12-20.mp4,5,1,5577.0,0.869,<0x0A>Вино▁GUSTARE,None,None,None,12345_678,...,None,None,None,1099,1099,,,12345_678,12345_678,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,25_12-20.mp4,255,1,2340.2,0.870,<0x0A>Продукты_название,None,None,None,Штрихкод,...,None,None,None,Цена_без_карты,Цена_с_картой,Промо_цена,Размер_скида,Код_артикул,Формат_кода,<0x0A><0x0A>Овощи
59,25_12-20.mp4,257,1,1414.9,0.924,<0x0A>Продукты_название,None,None,None,Штрихкод,...,None,None,None,Цена_без_карты,Цена_с_картой,Промо_цена,Размер_скида,Номер_артикул,Код_лайаут,▁Неясно<0x0A>Цена_без_карты:▁399₽<0x0A>Цена_с_...
60,25_12-20.mp4,258,1,970.1,0.512,<0x0A>Продукты_название,None,None,None,Штрихкод,...,None,None,None,Цена_без_карты,Цена_с_картой,Промо_цена,Размер_скида,Наименование,Код_лайаута,"Дата_печати<0x0A><0x0A>Обратите▁внимание,▁что▁..."
61,25_12-20.mp4,263,1,1043.3,0.845,<0x0A>Продукция_название,None,None,None,Штрихкод,...,None,None,None,Цена_без_карты,Цена_с_картой,Промо_цена,Размер_скида,Номер_артикул,Код_лайаута,Дата_печати


In [104]:
df_vlm.to_excel('vlm.xlsx', index = False)